In [ ]:
# !pip install kaggle
!pip install pandas numpy scikit-learn tensorflow optuna shap matplotlib seaborn
!pip install xgboost lightgbm catboost
!pip install -q optuna catboost eif xgboost lightgbm scikit-learn pandas numpy shap matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for eif
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (eif)


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Advanced Tree Models
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# Tuning & XAI
import optuna
import shap

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# 1. Install Hugging Face datasets library (No API key or login required)
!pip install -q datasets

import pandas as pd
from datasets import load_dataset

print("Downloading CICIDS2017 dataset from Hugging Face... (This will be much faster!)")

# 2. Load the machine_learning subset of CICIDS2017 directly into memory
dataset = load_dataset("bvsam/cic-ids-2017", "machine_learning", split="train")

# 3. Convert to a Pandas DataFrame
df = dataset.to_pandas()

# 4. Clean column names (Crucial: the dataset has weird spacing in column names like ' Label')
df.columns = df.columns.str.strip()

print(f"\nFull Dataset loaded. Shape: {df.shape}")

README.md: 0.00B [00:00, ?B/s]

machine_learning/Friday-WorkingHours-Aft(…):   0%|          | 0.00/22.2M [00:00<?, ?B/s]

machine_learning/Friday-WorkingHours-Aft(…):   0%|          | 0.00/15.7M [00:00<?, ?B/s]

machine_learning/Friday-WorkingHours-Mor(…):   0%|          | 0.00/19.4M [00:00<?, ?B/s]

machine_learning/Monday-WorkingHours.pca(…):   0%|          | 0.00/57.0M [00:00<?, ?B/s]

...

machine_learning/Thursday-WorkingHours-M(…):   0%|          | 0.00/17.2M [00:00<?, ?B/s]

machine_learning/Tuesday-WorkingHours.pc(…):   0%|          | 0.00/45.6M [00:00<?, ?B/s]

machine_learning/Wednesday-workingHours.(…):   0%|          | 0.00/67.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2830743 [00:00<?, ? examples/s]


Full Dataset loaded. Shape: (2830743, 79)


In [ ]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

def preprocess_data(df):
    print("Starting preprocessing...")

    # The exact unknown attacks chosen by the original authors
    unknown_attacks = ['DoS slowloris', 'DoS Slowhttptest', 'Bot']

    # 1. Separate the data into Benign, Known Attacks, and Unknown Attacks
    benign_df = df[df['Label'] == 'BENIGN']
    unknown_df = df[df['Label'].isin(unknown_attacks)]
    known_df = df[(df['Label'] != 'BENIGN') & (~df['Label'].isin(unknown_attacks))]

    print(f"Benign samples: {len(benign_df)}")
    print(f"Known attack samples: {len(known_df)}")
    print(f"Unknown attack samples: {len(unknown_df)}")

    # 2. Split 80% for training and 20% for testing
    b_train, b_test = train_test_split(benign_df, test_size=0.2, random_state=42)
    k_train, k_test = train_test_split(known_df, test_size=0.2, random_state=42)

    # --- BUILD THE 4 REQUIRED DATASETS ---

    # A. Supervised Training Set (80% Benign + 80% Known Attacks)
    train_sup = pd.concat([b_train, k_train]).sample(frac=1, random_state=42)

    # B. Unsupervised Training Set (80% Benign ONLY - for OCSVM and Isolation Forest)
    train_unsup = b_train.copy()

    # C. Overall Test Set (20% Benign + 20% Known + ALL Unknown)
    test_overall = pd.concat([b_test, k_test, unknown_df]).sample(frac=1, random_state=42)

    # D. Unknown Attack Test Set (ALL Unknown + Equal number of Benign traffic)
    num_unknown = len(unknown_df)
    b_test_sample = b_test.sample(n=min(num_unknown, len(b_test)), random_state=42)
    test_unknown = pd.concat([b_test_sample, unknown_df]).sample(frac=1, random_state=42)

    # Helper function to separate features (X) and labels (y)
    def get_X_y(data):
        # 0 for benign, 1 for malicious
        y = (data['Label'] != 'BENIGN').astype(int)
        X = data.drop(columns=['Label'])
        # Keep only numerical columns and handle infinite values
        X = X.select_dtypes(include=[np.number])
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
        return X, y

    # Extract X and y for all sets
    X_train_sup, y_train_sup = get_X_y(train_sup)
    X_train_unsup, y_train_unsup = get_X_y(train_unsup)
    X_test_all, y_test_all = get_X_y(test_overall)
    X_test_unk, y_test_unk = get_X_y(test_unknown)

    print("\nScaling features (StandardScaler)...")
    # 3. Scaling (crucial for Deep Learning and SVM)
    scaler = StandardScaler()
    X_train_sup_scaled = scaler.fit_transform(X_train_sup)
    X_train_unsup_scaled = scaler.transform(X_train_unsup)
    X_test_all_scaled = scaler.transform(X_test_all)
    X_test_unk_scaled = scaler.transform(X_test_unk)

    return (X_train_sup_scaled, y_train_sup, X_train_unsup_scaled, y_train_unsup,
            X_test_all_scaled, y_test_all, X_test_unk_scaled, y_test_unk, scaler)

# Run the function
(X_train_sup, y_train_sup, X_train_unsup, y_train_unsup,
 X_test_all, y_test_all, X_test_unk, y_test_unk, scaler) = preprocess_data(df)

print("\n--- Final Dataset Shapes ---")
print(f"Supervised Train Set: {X_train_sup.shape}")
print(f"Overall Test Set: {X_test_all.shape}")
print(f"Unknown Test Set: {X_test_unk.shape}")

Starting preprocessing...
Benign samples: 2273097
Known attack samples: 544385
Unknown attack samples: 13261


In [ ]:
def build_mlp(input_dim):
    model = Sequential([
        Dense(100, activation='relu', input_dim=input_dim),
        Dense(50, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    opt = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("Training MLP...")
mlp_model = build_mlp(X_train_sup.shape[1])
early_stop_mlp = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

mlp_history = mlp_model.fit(
    X_train_sup, y_train_sup,
    validation_split=0.2,
    epochs=100,
    batch_size=256,
    callbacks=[early_stop_mlp],
    verbose=1
)

Training MLP...
Epoch 1/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9620 - loss: 0.0934 - val_accuracy: 0.9718 - val_loss: 0.0610
Epoch 2/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9726 - loss: 0.0588 - val_accuracy: 0.9748 - val_loss: 0.0520
Epoch 3/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9756 - loss: 0.0527 - val_accuracy: 0.9775 - val_loss: 0.0481
Epoch 4/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9773 - loss: 0.0498 - val_accuracy: 0.9793 - val_loss: 0.0457
Epoch 5/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9782 - loss: 0.0477 - val_accuracy: 0.9797 - val_loss: 0.0441
Epoch 6/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9791 - loss: 0.0461 - val_accuracy: 0.9804 - val_loss: 0.0428
Epoch 7/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9797 - loss: 0.0447 - val_accuracy: 0.9816 - val_loss: 0.0413
Epoch 8/100
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        alpha_t = y_true * alpha + (tf.ones_like(y_true) - y_true) * (1 - alpha)
        p_t = y_true * y_pred + (tf.ones_like(y_true) - y_true) * (1 - y_pred)
        loss = - alpha_t * tf.math.pow((tf.ones_like(y_true) - p_t), gamma) * tf.math.log(p_t)
        return tf.reduce_mean(loss)
    return focal_loss_fn

def build_cnn(input_shape):
    model = Sequential([
        Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Conv1D(64, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss=focal_loss(), metrics=['accuracy'])
    return model

print("Training 1D-CNN...")
# Reshape for Conv1D: (samples, features, 1)
X_train_cnn = X_train_sup.reshape((X_train_sup.shape[0], X_train_sup.shape[1], 1))
cnn_model = build_cnn((X_train_cnn.shape[1], 1))
early_stop_cnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

class_weights = {0: 1.0, 1: 5.0}

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_sup,
    validation_split=0.2,
    epochs=50,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[early_stop_cnn],
    verbose=1
)

Training 1D-CNN...
Epoch 1/50


In [ ]:
print("Training OCSVM...")
ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(X_train_svm)

print("Training LOF...")
lof = LocalOutlierFactor(n_neighbors=80, novelty=True, metric='minkowski', p=2, leaf_size=80)
lof.fit(X_train_unsup)

# Helper for unsupervised prediction mapping: sklearn returns 1 (normal) and -1 (anomaly).
# We need to map to 0 (benign) and 1 (malicious).
def map_anomaly_preds(preds):
    return np.where(preds == 1, 0, 1)

XGBOOST

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'tree_method': 'hist', # crucial for full dataset speed
        'random_state': 42
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=10)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
    return f1_score(y_v, model.predict(X_v))

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=5)
best_xgb = xgb.XGBClassifier(**study_xgb.best_params, tree_method='hist', random_state=42)
print("Training Best XGBoost on FULL Supervised Set...")
best_xgb.fit(X_train_sup, y_train_sup)


LightGBM

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 31, 150),
        'random_state': 42,
        'n_jobs': -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_t, y_t, eval_set=[(X_v, y_v)], callbacks=[lgb.early_stopping(10, verbose=False)])
    return f1_score(y_v, model.predict(X_v))

print("\nTuning LightGBM...")
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=5)
best_lgb = lgb.LGBMClassifier(**study_lgb.best_params, random_state=42)
print("Training Best LightGBM on FULL Supervised Set...")
best_lgb.fit(X_train_sup, y_train_sup)

Catboost

In [ ]:
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 300),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'random_seed': 42,
        'verbose': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_t, y_t, eval_set=(X_v, y_v), early_stopping_rounds=10)
    return f1_score(y_v, model.predict(X_v))

print("\nTuning CatBoost...")
study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=5)
best_cat = CatBoostClassifier(**study_cat.best_params, random_seed=42, verbose=0)
print("Training Best CatBoost on FULL Supervised Set...")
best_cat.fit(X_train_sup, y_train_sup)

RandomForest

In [ ]:
best_rf = RandomForestClassifier(n_estimators=150, max_depth=15, n_jobs=-1, random_state=42)
best_rf.fit(X_train_sup, y_train_sup)

# ANAMOLY DETECTION

XGBOOST - UNSUPERVISED

In [ ]:
from sklearn.ensemble import IsolationForest
import eif as iso # Extended Isolation Forest
import numpy as np

print("--- PREPARING UNSUPERVISED TREE MODELS ---")

# ---------------------------------------------------------
# 1. UNSUPERVISED XGBOOST (Synthetic Outlier Generation)
# This forces XGBoost to learn the exact boundary of benign traffic
# ---------------------------------------------------------
print("\nPreparing Synthetic Noise for Unsupervised XGBoost...")
# Generate synthetic noise equal to 20% of benign traffic to act as pseudo-anomalies
n_syn = int(len(X_train_unsup) * 0.20)
min_vals, max_vals = X_train_unsup.min(axis=0), X_train_unsup.max(axis=0)

# Generate uniform noise within feature bounds
synthetic_noise = np.random.uniform(low=min_vals, high=max_vals, size=(n_syn, X_train_unsup.shape[1]))

# Combine Benign (Label 0) and Noise (Label 1)
X_syn_train = np.vstack([X_train_unsup, synthetic_noise])
y_syn_train = np.hstack([np.zeros(len(X_train_unsup)), np.ones(n_syn)])

print("Training Unsupervised XGBoost (Boundary Detector)...")
# Using hist and n_jobs for massive dataset speed
unsup_xgb = xgb.XGBClassifier(n_estimators=200, max_depth=6, tree_method='hist', n_jobs=-1, random_state=42)
unsup_xgb.fit(X_syn_train, y_syn_train)

ISOLATION FOREST

In [ ]:
print("\nTraining Isolation Forest on FULL Benign Dataset...")
iso_forest = IsolationForest(n_estimators=200, max_samples='auto', contamination=0.05, n_jobs=-1, random_state=42)
iso_forest.fit(X_train_unsup)


# 3. EXTENDED ISOLATION FOREST (EIF)


In [ ]:
print("\nTraining Extended Isolation Forest (EIF)...")
# EIF builds hyperplanes with varying slopes, often outperforming standard IF.
# It is computationally heavy, so we limit tree count slightly for the full dataset.
eif_model = iso.iForest(X_train_unsup, ntrees=100, sample_size=256, ExtensionLevel=1)

# cHECK IF BEAT BENCHMARK

In [ ]:
import pandas as pd

# Paper Benchmarks
BENCHMARK_SUP_F1 = 0.9446  # MLP on Known Attacks
BENCHMARK_UNSUP_F1 = 0.7575 # OCSVM on Unknown Attacks

# Helper for standard scikit-learn anomaly outputs (-1 = anomaly, 1 = normal)
def format_anomaly_preds(preds):
    return np.where(preds == 1, 0, 1)

print("="*60)
print("🏆 SUPERVISED EVALUATION (OVERALL TEST SET) 🏆")
print("Goal: Beat MLP F1-Score of {:.4f}".format(BENCHMARK_SUP_F1))
print("="*60)

sup_models = {
    "XGBoost": best_xgb,
    "LightGBM": best_lgb,
    "CatBoost": best_cat,
    "Random Forest": best_rf
}

for name, model in sup_models.items():
    preds = model.predict(X_test_all)
    f1 = f1_score(y_test_all, preds)
    acc = accuracy_score(y_test_all, preds)
    status = "✅ BEAT BENCHMARK!" if f1 > BENCHMARK_SUP_F1 else "❌ Failed"
    print(f"{name:15s} | Accuracy: {acc:.4f} | F1-Score: {f1:.4f} | {status}")


print("\n" + "="*60)
print("🛡️ UNSUPERVISED EVALUATION (UNKNOWN / ZERO-DAY ATTACKS) 🛡️")
print("Goal: Beat OCSVM F1-Score of {:.4f}".format(BENCHMARK_UNSUP_F1))
print("="*60)

# Evaluate Unsupervised XGBoost
unsup_xgb_probs = unsup_xgb.predict_proba(X_test_unk)[:, 1]
# We use a strict threshold since anomalies are sparse
unsup_xgb_preds = (unsup_xgb_probs > 0.85).astype(int)
unsup_xgb_f1 = f1_score(y_test_unk, unsup_xgb_preds)

# Evaluate Isolation Forest
iso_preds = format_anomaly_preds(iso_forest.predict(X_test_unk))
iso_f1 = f1_score(y_test_unk, iso_preds)

# Evaluate Extended Isolation Forest
# EIF returns anomaly scores. We need to set a threshold (usually ~0.50)
eif_scores = eif_model.compute_paths(X_test_unk)
eif_preds = (eif_scores > 0.55).astype(int)
eif_f1 = f1_score(y_test_unk, eif_preds)

unsup_results = {
    "Unsup XGBoost": unsup_xgb_f1,
    "Isolation Forest": iso_f1,
    "Extended IF": eif_f1
}

for name, f1 in unsup_results.items():
    status = "✅ BEAT BENCHMARK!" if f1 > BENCHMARK_UNSUP_F1 else "❌ Failed"
    print(f"{name:18s} | F1-Score: {f1:.4f} | {status}")

print("\nDetailed Report of Best Unsupervised Model:")
best_unsup_name = max(unsup_results, key=unsup_results.get)
if best_unsup_name == "Unsup XGBoost":
    print(classification_report(y_test_unk, unsup_xgb_preds))
elif best_unsup_name == "Isolation Forest":
    print(classification_report(y_test_unk, iso_preds))
else:
    print(classification_report(y_test_unk, eif_preds))